- **거리 측정 방법**
    1. **유클리드 거리(Euclidean Distance):** 두 점 사이의 직선 거리로 가장 많이 사용된다.
        
        $$
        d(p, q) = \sqrt{\sum_{i=1}^n (p_i - q_i)^2}
        $$
        
        - 기본 개념
            - **두 점 p와 q 사이의 거리(distance)** 를 계산

In [ ]:
import math

p = [1, 2]
q = [4, 6]

distance = math.dist(p, q)
print(distance)  

5.0


2. **맨해튼 거리(Manhattan Distance)**: 절대 거리의 합으로 도시 블록 구조와 유사하다.
    
    $$
    d(p, q) = \sum_{i=1}^n |p_i - q_i|
    $$
    
    - 기본 개념
        - 각 좌표 축에서 떨어진 거리(절댓값)를 모두 더한 값이 두 점 사이의 L1 거리이다.


In [ ]:
p = [1, 2]
q = [4, 6]

manhattan = sum(abs(pi - qi) for pi, qi in zip(p, q))
print(manhattan)   

7


3. **코사인 유사도(Cosine Similarity)**: 벡터 간의 방향 유사도를 기반으로 한다.
    
    $$
    \text{Cosine Similarity} = \frac{\vec{p} \cdot \vec{q}}{\|\vec{p}\| \|\vec{q}\|}
    $$
    
    - 기본 개념
        - 두 벡터가 얼마나 같은 방향을 향하고 있는지 측정하는 값. 벡터의 크기보다 ‘방향’이 얼마나 유사한지가 중요할 때 사용한다.
        - 값의 범위 1 : 완전히 같은 방향 / 0 : 직각(전혀 관련 없음) / -1 : 완전히 반대 방향
        - 문서 벡터(단어 빈도) 비교, 추천 시스템, 임베딩 비교 등에 매우 자주 사용됨.

- **비용 함수: 로그 손실(Log Loss)**
    
    로지스틱 회귀는 예측 결과를 `0~1` 사이의 확률로 출력하므로, 예측 확률이 실제 정답과 얼마나 다른지 측정하기 위해 **Log Loss**를 사용한다.
    
    $$
    J(\theta) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(h_\theta(x^{(i)})) + (1 - y^{(i)}) \log(1 - h_\theta(x^{(i)})) \right]
    $$
    
    - $m$ : 데이터 샘플의 수
    - $y^{(i)}$ : 실제 정답값 `0` 또는 `1`
    - $h_\theta(x^{(i)})$ : 모델이 예측한 클래스 1의 확률
    - $h_\theta(x^{(i)})$을 간단히 $p$라고 생각하면, $p = h_\theta(x)$
        
        → 즉, $p$는 모델이 예측한 **클래스 1일 확률**이다.
        
    - 실제 정답값은 `0` 또는 `1` 값만 가지기 때문에
        - 실제 정답이 1인 경우
            
            $y = 1$ : 이때는 $(1-y)$가 0이 되므로 뒤쪽 항이 사라진다.
            
            $Loss = -\log(p)$ : 정답이 1일 때는 모델이 예측한 확률 $p$가 1에 가까울수록 손실이 작다.
            
            - 실제값 1, 예측 확률 0.99 : 손실 작음
            - 실제값 1, 예측 확률 0.01 : 손실 큼
        - 실제 정답이 0인 경우
            
            $y = 0$ : 이때는 앞쪽 항이 사라진다.
            
            $Loss = -\log(1-p)$ : 정답이 0일 때는 모델이 예측한 확률 $p$가 0에 가까울수록 손실이 작다.
            
            - 실제값 0, 예측 확률 0.01 → 손실 작음
            - 실제값 0, 예측 확률 0.99 → 손실 큼

In [4]:
import numpy as np

# 간단한 데이터 (x: 특성, y: 레이블)
x = np.array([0.2, 0.8, 1.5])                     # 입력 특성 값들
y = np.array([0, 1, 1])                           # 실제 정답 레이블 (0/1)

theta = 1.0                                       # 임의의 가중치값 θ
z = theta * x                                     # 선형 결합 z = θx
h = 1 / (1 + np.exp(-z))                          # 시그모이드 함수 → 예측 확률 h(x)

# Log Loss 계산
log_loss = -np.mean(y*np.log(h) + (1-y)*np.log(1-h))   # 로지스틱 회귀의 비용 함수(Log Loss)
print(round(log_loss, 3))              # 예: 0.391       # 계산된 손실값 출력

# 예측 (threshold=0.5)
pred = (h >= 0.5).astype(int)                      # 0.5 이상이면 1, 아니면 0으로 분류
print(pred)                             # 예: [0 1 1]    # 최종 예측 결과

0.457
[1 1 1]


### 04-03-04. 다중 레이블 분류 (Multilabel Classification)

- **다중 레이블 분류**란 하나의 데이터 포인트가 여러 클래스에 동시에 속할 수 있는 문제이다.
    - 예: 영화 장르 분류(코미디, 액션, 드라마)
- **로지스틱 회귀의 적용**
    - 각 레이블에 대해 독립적인 로지스틱 회귀 모델을 학습한다.
    - 출력: 각 레이블에 속할 확률 벡터
- **손실 함수 :** 각 레이블에 대해 개별적으로 손실을 계산하고, 이를 합산한다.

In [5]:
import numpy as np

# 입력 벡터
x = np.array([1.2, 0.5])

# 각 레이블별 가중치 (3개의 레이블: 액션, 코미디, 드라마)
theta = np.array([
    [1.0, -0.5],   # 액션
    [-0.3, 1.2],   # 코미디
    [0.8, 0.8]     # 드라마
])

# 레이블별 선형 조합 + 시그모이드
z = theta @ x
sigmoid = 1 / (1 + np.exp(-z))

print(np.round(sigmoid, 3))   # 예: [0.777 0.622 0.845]

[0.721 0.56  0.796]


| 구분 | 이진 분류 (Binary Classification) | 다중 분류 (Multiclass Classification) | 다중 레이블 분류 (Multilabel Classification) | 다중 출력 분류 (Multi-output Classification) |
| --- | --- | --- | --- | --- |
| **예측 대상 개수** | 1개 | 1개 | 여러 개 | 여러 개 |
| **클래스 관계** | 둘 중 하나 (0 / 1) | 여러 클래스 중 **하나만 선택** | 여러 클래스 **동시에 선택 가능** | 출력 변수마다 클래스 존재 |
| **대표 예시** | 스팸 / 정상 메일 | 숫자(0~9) 분류 | 사진 태그 (사람, 동물, 자동차) | 한 사람의 혈액형 + 성별 + 질병유무 |
| **주요 모델** | 로지스틱 회귀 | 소프트맥스 로지스틱 회귀 | 다중 이진 분류기 조합 | 여러 분류기를 묶은 모델 |
| **활성화 함수** | Sigmoid | Softmax | Sigmoid (각 클래스별) | 출력별로 다름 |
| **출력 값 형태** | 확률 1개 | 클래스별 확률 벡터 | 클래스별 확률 벡터 | 출력 변수별 예측값 |
| **출력 범위** | 0 ~ 1 | 각 클래스 0~1 (합 = 1) | 각 클래스 0~1 (합 X) | 출력마다 다름 |
| **분류 기준** | 임계값 (보통 0.5) | 가장 큰 확률 | 클래스별 임계값 적용 | 출력별 기준 적용 |
| **대표 손실 함수** | Binary Cross-Entropy | Categorical Cross-Entropy | Binary Cross-Entropy | 출력별 손실 합 |

## 04-04. 모델 성능 측정

### 04-04-01. 정확도 (Accuracy)

<aside>
💡 모델이 예측한 결과와 실제 데이터의 일치도를 나타내는 지표로, 분류 모델의 경우 주로 정확도를 측정하여 평가한다.

</aside>
TN(True Negative), FP(False Positive), FN(False Negative), TP(True Positive)


- 정확도 공식
    
    $$
    Accuracy = \frac{TP + TN}{TP + TN + FP + FN}
    $$
    
    - $TP$는 참 긍정(True Positive), $TN$은 참 부정(True Negative), $FP$는 거짓 긍정(False Positive), $FN$은 거짓 부정(False Negative)을 의미한다.

In [6]:
import numpy as np

y_true = np.array([1, 0, 1, 1, 0])   # 실제값
y_pred = np.array([1, 0, 0, 1, 0])   # 예측값

TP = np.sum((y_true == 1) & (y_pred == 1))
TN = np.sum((y_true == 0) & (y_pred == 0))
FP = np.sum((y_true == 0) & (y_pred == 1))
FN = np.sum((y_true == 1) & (y_pred == 0))

accuracy = (TP + TN) / len(y_true)
print("Accuracy:", round(accuracy, 3))   # 예: 0.8

Accuracy: 0.8


- 정밀도 (Precision)
    
    > TP / (FP + TP) : 모델이 양성으로 예측한 것들 중에서 실제 양성인 비율
    > 
    - 해석 : 모델의 탐지 결과를 얼마나 믿을 수 있는가?
    - 정밀도가 높아야 하는 경우 : 스팸 메일 필터링

- 재현율 (Recall)
    
    > TP / (FN + TP) : 실제로 양성 중에서 모델이 양성이라고 예측한 비율
    > 
    - 해석 : 모델이 데이터 내에서 이상을 탐지했는지?
    - 재현율이 높아야 하는 경우 : 암 여부 판별

- F1 스코어
    
    $$
    F1\ Score = 2 \times \frac{Precision \times Recall}{Precision + Recall}
    $$
    
    > 정밀도 + 재현율
    >- 정밀도와 재현율의 관계 : 서로 상충되는 관계
    - 정밀도가 높으면 재현율이 떨어진다.
    - 재현율이 높아지면 정밀도가 낮아진다.
    - F1 스코어는 이 두개의 지표의 조화평

- **AUC와 ROC**
    - **ROC Curve(**Receiver Operating Characteristic Curve**)**는 참 긍정 비율(True Positive Rate, TPR)과 거짓 긍정 비율(False Positive Rate, FPR)의 관계를 나타내는 곡선이다.
    - 참 긍정 비율(TPR)은 재현율(Recall)과 같으며, 거짓 긍정 비율(FPR)은 모델이 잘못 양성으로 예측한 비율을 의미한다.
        
        $$
        TPR = \frac{TP}{TP + FN}, \quad FPR = \frac{FP}{FP + TN}
        $$
        
    - ROC Curve는 **FPR**이 증가할 때 **TPR**이 얼마나 증가하는지 보여준다. 이 곡선 아래 면적이 클수록, 즉 **AUC** 값이 클수록 모델의 성능이 좋다는 것을 의미한다.
    - **AUC**는 ROC 곡선 아래의 면적을 의미하며, 0과 1 사이의 값을 가진다.